In [3]:
!git clone https://github.com/sharul-ayub/malaysia-bank-employee-sentiment-analysis.git

Cloning into 'malaysia-bank-employee-sentiment-analysis'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 41 (delta 9), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 289.92 KiB | 12.60 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [4]:
%cd malaysia-bank-employee-sentiment-analysis

/content/malaysia-bank-employee-sentiment-analysis


# 01 — Data Preparation and Sentence Splitting

This notebook covers:

1. Load the public-safe source dataset.
2. Filter the intended review period.
3. Reshape review text fields into one NLP text column.
4. Split review text into sentence-level rows.
5. Remove one-word fragments.
6. Export the sentence-level dataset for manual inspection.

> **Project period:** The project use data period from 2024–2026. The raw dataset contain period 2012–2026, so this notebook uses configurable `START_YEAR = 2024` and `END_YEAR = 2026`.

In [26]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(".")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank_employee_reviews_raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Load dataset

In [18]:
df = pd.read_csv(RAW_DATA_PATH)
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (1954, 19)


,id,company_name,review_date,review_title,review_body_text,overall_review_rating,job_title,location,employee_flag,rating_work_life_balance,rating_compensation_benefits,rating_job_security_advancement,rating_management,rating_culture_values,pros_text,cons_text,page_number,start_offset,source_url
0,1,Cimb-Group,8/4/2026,Clear object,Good company and exposure for junior level. Pr...,4,Manager,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...
1,2,HSBC,26/3/2026,best place to work,"work life balance, good elearning, internaltio...",4,manager,Malaysia,former,0,0,0,0,0,"work life balance, good elearning",low increament,1,0,https://malaysia.indeed.com/cmp/HSBC/reviews
2,3,Cimb-Group,20/3/2026,Productive workplace,"brilliant training initiatives, cool projects ...",4,Fusion programme,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...
3,4,bank islam,8/3/2026,Comprehensive benefits with concerns over oper...,I spent more than 10 years at Bank Islam and f...,3,NaN,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Bank-Islam/rev...
4,5,Cimb-Group,7/3/2026,Productive and fun workplace,Flexible working hours and supportive colleagu...,4,Credit Officer,Kuala Lumpur,current,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...


## Filter the review period

The source dates are day/month/year, so `dayfirst=True` is used.

In [19]:
START_YEAR = 2024
END_YEAR = 2026

df["review_date"] = pd.to_datetime(
    df["review_date"],
    errors="coerce",
    dayfirst=True
)

df_filtered = df[
    df["review_date"].dt.year.between(START_YEAR, END_YEAR)
].copy()

print("Original rows:", len(df))
print("Filtered rows:", len(df_filtered))
print("Date range:",
      df_filtered["review_date"].min(),
      "to",
      df_filtered["review_date"].max())
display(df_filtered.head())

Original rows: 1954
Filtered rows: 291
Date range: 2024-01-07 00:00:00 to 2026-04-08 00:00:00


,id,company_name,review_date,review_title,review_body_text,overall_review_rating,job_title,location,employee_flag,rating_work_life_balance,rating_compensation_benefits,rating_job_security_advancement,rating_management,rating_culture_values,pros_text,cons_text,page_number,start_offset,source_url
0,1,Cimb-Group,2026-04-08,Clear object,Good company and exposure for junior level. Pr...,4,Manager,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...
1,2,HSBC,2026-03-26,best place to work,"work life balance, good elearning, internaltio...",4,manager,Malaysia,former,0,0,0,0,0,"work life balance, good elearning",low increament,1,0,https://malaysia.indeed.com/cmp/HSBC/reviews
2,3,Cimb-Group,2026-03-20,Productive workplace,"brilliant training initiatives, cool projects ...",4,Fusion programme,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...
3,4,bank islam,2026-03-08,Comprehensive benefits with concerns over oper...,I spent more than 10 years at Bank Islam and f...,3,NaN,NaN,former,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Bank-Islam/rev...
4,5,Cimb-Group,2026-03-07,Productive and fun workplace,Flexible working hours and supportive colleagu...,4,Credit Officer,Kuala Lumpur,current,0,0,0,0,0,NaN,NaN,1,0,https://malaysia.indeed.com/cmp/Cimb-Group/rev...


## Convert review text columns into NLP rows

In [20]:
text_fields = [
    "review_title",
    "review_body_text",
    "pros_text",
    "cons_text"
]

nlp_df = df_filtered.melt(
    id_vars=[c for c in df_filtered.columns if c not in text_fields],
    value_vars=text_fields,
    var_name="text_content",
    value_name="text"
)

nlp_df = nlp_df.dropna(subset=["text"])
nlp_df = nlp_df[nlp_df["text"].astype(str).str.strip().ne("")].copy()

nlp_df["text_content"] = pd.Categorical(
    nlp_df["text_content"],
    categories=text_fields,
    ordered=True
)

nlp_df = (
    nlp_df
    .sort_values(["id", "text_content"])
    .reset_index(drop=True)
)

nlp_df.insert(0, "review_row_id", range(1, len(nlp_df) + 1))

columns_to_keep = [
    "review_row_id",
    "id",
    "company_name",
    "text_content",
    "text"
]
nlp_df = nlp_df[columns_to_keep]

print("NLP text rows created:", len(nlp_df))
display(nlp_df.head())

NLP text rows created: 640


,review_row_id,id,company_name,text_content,text
0,1,1,Cimb-Group,review_title,Clear object
1,2,1,Cimb-Group,review_body_text,Good company and exposure for junior level. Pr...
2,3,2,HSBC,review_title,best place to work
3,4,2,HSBC,review_body_text,"work life balance, good elearning, internaltio..."
4,5,2,HSBC,pros_text,"work life balance, good elearning"


## Sentence splitting rules

In [21]:
SPLIT_TOKEN = "<SPLIT_SENT>"
DOT_TOKEN = "<DOT_KEEP>"
DEC_TOKEN = "<DEC_KEEP>"
PAREN_HYPHEN_TOKEN = "<PAREN_HYPHEN_KEEP>"
PROCON_DASH_TOKEN = "<PROCON_DASH_KEEP>"
DAY_RANGE_HYPHEN_TOKEN = "<DAY_RANGE_HYPHEN_KEEP>"
LABEL_COLON_DASH_TOKEN = "<LABEL_COLON_DASH_KEEP>"
CONJ_COMMA_DASH_TOKEN = "<CONJ_COMMA_DASH_KEEP>"

ABBREVIATIONS = [
    "e.g.", "i.e.", "mr.", "mrs.", "ms.", "dr.", "prof.",
    "inc.", "ltd.", "co.", "corp.", "u.s.", "u.k.", "sdn.", "bhd."
]

def clean_space(text):
    return " ".join((text or "").split())

def layer1_prepare_and_split(text):
    t = (text or "").replace("\r\n", "\n").replace("\r", "\n")

    # Remove emoji/pictographs
    t = re.sub(
        r"[\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
        r"\U0001F680-\U0001F6FF\U0001F700-\U0001F77F"
        r"\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF"
        r"\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
        r"\U00002700-\U000027BF\U0001F1E6-\U0001F1FF]+",
        "",
        t,
    )

    # Split bullets and numbered lists
    t = re.sub(r"^\s*[-*•]+\s*", f"{SPLIT_TOKEN} ", t)
    t = re.sub(r"\n\s*[-*•]+\s*", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"(?:(?<=^)|(?<=\s))-(?=[A-Za-z])", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"\n\s*\d{1,3}[\.)]\s*", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"(?:^|(?<=\s))\d{1,3}[\.)](?=\s|[A-Za-z])\s*", f" {SPLIT_TOKEN} ", t)

    # Split line breaks and semicolons
    t = re.sub(r"\n+", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r";+\s*", f" {SPLIT_TOKEN} ", t)

    # Protect decimals
    t = re.sub(r"(\d)\.(\d)", rf"\1{DEC_TOKEN}\2", t)
    return t

def protect_abbreviations(text):
    out = text
    for abbr in ABBREVIATIONS:
        escaped = re.escape(abbr)
        repl = abbr.replace(".", DOT_TOKEN)
        out = re.sub(escaped, repl, out, flags=re.IGNORECASE)
    return out

def protect_hyphen_in_parentheses(text):
    return re.sub(
        r"\([^()]*\)",
        lambda m: m.group(0).replace("-", PAREN_HYPHEN_TOKEN),
        text
    )

def protect_pros_cons_dash(text):
    pattern = r"(?i)\b(pros?|cons?)\s*:?\s*(?:-\s*){1,3}"
    return re.sub(pattern, lambda m: f"{m.group(1)} {PROCON_DASH_TOKEN} ", text)

def protect_day_ranges(text):
    pattern = (
        r"(?i)\b("
        r"mon(?:day)?|tue(?:s|sday)?|wed(?:nesday)?|thu(?:r|rs|rsday)?|"
        r"fri(?:day)?|sat(?:urday)?|sun(?:day)?"
        r")\s*-\s*("
        r"mon(?:day)?|tue(?:s|sday)?|wed(?:nesday)?|thu(?:r|rs|rsday)?|"
        r"fri(?:day)?|sat(?:urday)?|sun(?:day)?"
        r")\b"
    )
    return re.sub(
        pattern,
        lambda m: f"{m.group(1)} {DAY_RANGE_HYPHEN_TOKEN} {m.group(2)}",
        text
    )

def protect_label_colon_dash(text):
    pattern = r"\b([A-Za-z][A-Za-z/& ]{1,40})\s*:\s*-\s+"
    return re.sub(
        pattern,
        lambda m: f"{m.group(1).strip()}: {LABEL_COLON_DASH_TOKEN} ",
        text
    )

def split_before_label_colon_dash(text):
    pattern = (
        r"(?i)\s+(?="
        r"(?:what|the|why|how|my|overall|advice|areas|benefit|benefits|pros?|cons?)"
        r"(?:\s+[A-Za-z][A-Za-z/&]*){0,8}"
        r"\s*:\s*<LABEL_COLON_DASH_KEEP>\s*"
        r")"
    )
    return re.sub(pattern, f" {SPLIT_TOKEN} ", text)

def protect_conjunction_comma_dash(text):
    pattern = r"(?i)\b(but|and|so|or|however|therefore),\s*-\s+"
    return re.sub(
        pattern,
        lambda m: f"{m.group(1)}, {CONJ_COMMA_DASH_TOKEN} ",
        text
    )

def split_inline_dash_bullets(text):
    return re.sub(r"\s-\s(?=(?:[A-Za-z]|\())", f" {SPLIT_TOKEN} ", text)

def layer2_split(text):
    t = protect_hyphen_in_parentheses(text)
    t = protect_pros_cons_dash(t)
    t = protect_day_ranges(t)
    t = protect_label_colon_dash(t)
    t = protect_conjunction_comma_dash(t)
    t = split_before_label_colon_dash(t)
    t = split_inline_dash_bullets(t)

    t = re.sub(
        r"(?i)\s+(?=(?:pro|pros|con|cons)\s*(?::|-))",
        f" {SPLIT_TOKEN} ",
        t
    )

    t = protect_abbreviations(t)

    t = re.sub(r"([!?]+)(\s+|$)", rf"\1 {SPLIT_TOKEN} ", t)
    t = re.sub(r"((?<!\.)\.(?!\.))(\s+|$)", rf"\1 {SPLIT_TOKEN} ", t)
    t = re.sub(r"([.!?])([A-Za-z])", rf"\1 {SPLIT_TOKEN} \2", t)

    t = t.replace(DOT_TOKEN, ".")
    t = t.replace(DEC_TOKEN, ".")
    t = t.replace(PAREN_HYPHEN_TOKEN, "-")
    t = t.replace(PROCON_DASH_TOKEN, "-")
    t = t.replace(DAY_RANGE_HYPHEN_TOKEN, "-")
    t = t.replace(LABEL_COLON_DASH_TOKEN, "-")
    t = t.replace(CONJ_COMMA_DASH_TOKEN, "-")

    parts = [clean_space(p) for p in t.split(SPLIT_TOKEN)]
    parts = [p for p in parts if p and p not in {"-", "*", "•"}]

    out = []
    for p in parts:
        if p.count(">") >= 2:
            out.extend([
                clean_space(x)
                for x in re.split(r"\s*>\s*", p)
                if clean_space(x)
            ])
        else:
            out.append(p)

    return out

def split_text_to_sentences(text):
    if pd.isna(text) or not str(text).strip():
        return []

    stage1 = layer1_prepare_and_split(str(text))
    return layer2_split(stage1)

## Create sentence-level dataframe

In [22]:
sentence_rows = []

for _, row in nlp_df.iterrows():
    sentences = split_text_to_sentences(row["text"])

    for sentence_id, sentence in enumerate(sentences, start=1):
        sentence_rows.append({
            "review_row_id": row["review_row_id"],
            "id": row["id"],
            "company_name": row["company_name"],
            "text_content": row["text_content"],
            "sentence_id": sentence_id,
            "sentence_text": sentence,
            "original_text": row["text"]
        })

sentence_df = pd.DataFrame(sentence_rows)

sentence_df = (
    sentence_df
    .sort_values(["id", "review_row_id", "text_content", "sentence_id"])
    .reset_index(drop=True)
)

print("Original NLP rows:", len(nlp_df))
print("Sentence-level rows:", len(sentence_df))
display(sentence_df.head())

Original NLP rows: 640
Sentence-level rows: 1462


,review_row_id,id,company_name,text_content,sentence_id,sentence_text,original_text
0,1,1,Cimb-Group,review_title,1,Clear object,Clear object
1,2,1,Cimb-Group,review_body_text,1,Good company and exposure for junior level.,Good company and exposure for junior level. Pr...
2,2,1,Cimb-Group,review_body_text,2,Promotion can be very fast to some work functi...,Good company and exposure for junior level. Pr...
3,3,2,HSBC,review_title,1,best place to work,best place to work
4,4,2,HSBC,review_body_text,1,"work life balance, good elearning, internaltio...","work life balance, good elearning, internaltio..."


## Remove single-word fragments

In [23]:
rows_before = len(sentence_df)

sentence_df = sentence_df[
    sentence_df["sentence_text"].astype(str).str.split().str.len() > 1
].reset_index(drop=True)

rows_after = len(sentence_df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed:", rows_before - rows_after)

Rows before: 1462
Rows after: 1416
Removed: 46


## Export for manual inspection

Open the exported CSV, inspect the sentence boundaries, correct any incorrect splits, and save the manually checked file as:

`data/processed/02_sentence_level_manual_inspection.csv`

In [27]:
sentence_output = PROCESSED_DIR / "01_sentence_level.csv"
manual_output = PROCESSED_DIR / "02_sentence_level_manual_inspection.csv"

sentence_df.to_csv(sentence_output, index=False, encoding="utf-8-sig")
sentence_df.to_csv(manual_output, index=False, encoding="utf-8-sig")

print("Saved:", sentence_output)
print("After manual inspection, save your checked file as:")
print(manual_output)

Saved: data/processed/01_sentence_level.csv
After manual inspection, save your checked file as:
data/processed/02_sentence_level_manual_inspection.csv
